# Segment mix comparison

Monte Carlo comparison of **default** (LA-style) vs **balanced** (more pre-approved / self-cert) permit mixes at **6,571 permits**.

1. **Setup** — parameters and helper functions
2. **Run and compare** — one multi-run cell per scenario, then segment box plots

For the staffing × volume grid under `lognormal_180`, see [`run_simulation_with_segments_cases.ipynb`](run_simulation_with_segments_cases.ipynb).


## 1. Setup

In [ ]:
import sys
from pathlib import Path

_repo = Path.cwd()
if not (_repo / "run_simulation.py").exists():
    _repo = _repo.parent
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from run_simulation import run_multiple_simulations
from visualize_permits import plot_total_time_by_segment

NUM_PERMITS = 6571
RANDOM_SEED = 42
N_RUNS = 10

SCENARIO_DEFAULT = {
    "name": "Default",
    "sequential": "standard",
    # permit_mix="la" is the model default (~90% custom, 2% pre-approved, 8% self-cert)
}

SCENARIO_BALANCED = {
    "name": "Balanced",
    "sequential": "standard",
    "pct_pre_approved": 0.5,
    "pct_custom": 0.25,
    "pct_self_cert": 0.25,
    "pct_like_for_like": 0.8,
}


def _print_summary(name: str, permits: list) -> None:
    total_times = np.array(
        [
            p.ready_for_construction - p.created_at
            for p in permits
            if getattr(p, "ready_for_construction", None) is not None
        ]
    )
    if total_times.size == 0:
        print(f"{name}: no completed permits")
        return
    print(
        f"{name}: n={len(total_times):,}, "
        f"mean={total_times.mean():.1f} d, median={np.median(total_times):.1f} d"
    )

## 2. Run and compare

### Default mix

In [ ]:
print(f"Running Default: {N_RUNS} simulations ({NUM_PERMITS:,} permits each)...")

default_results = run_multiple_simulations(
    n_runs=N_RUNS,
    num_permits=NUM_PERMITS,
    base_seed=RANDOM_SEED,
    scenario_params_list=[SCENARIO_DEFAULT],
    collect_permits=True,
)

all_default_permits: list = []
for res in default_results:
    all_default_permits.extend(res.get("permits", []))

_print_summary("Default", all_default_permits)

In [ ]:
fig, ax = plot_total_time_by_segment(all_default_permits, figsize=(10, 7))
if ax is not None:
    ax.set_title(
        f"Default mix — disaster to construction by segment\n"
        f"({N_RUNS} runs × {NUM_PERMITS:,} permits, pooled)",
        fontsize=12,
    )
plt.show()

### Balanced mix

In [ ]:
print(f"Running Balanced: {N_RUNS} simulations ({NUM_PERMITS:,} permits each)...")

balanced_results = run_multiple_simulations(
    n_runs=N_RUNS,
    num_permits=NUM_PERMITS,
    base_seed=RANDOM_SEED,
    scenario_params_list=[SCENARIO_BALANCED],
    collect_permits=True,
)

all_balanced_permits: list = []
for res in balanced_results:
    all_balanced_permits.extend(res.get("permits", []))

_print_summary("Balanced", all_balanced_permits)

In [ ]:
fig, ax = plot_total_time_by_segment(all_balanced_permits, figsize=(10, 7))
if ax is not None:
    ax.set_title(
        f"Balanced mix — disaster to construction by segment\n"
        f"({N_RUNS} runs × {NUM_PERMITS:,} permits, pooled)",
        fontsize=12,
    )
plt.show()